# ⚡ Simulação EnergyPlus: VS Code + Google Colab Extension

Este notebook foi otimizado para execução via **VS Code com a extensão Google Colab**.

## 🚀 Fluxo de Trabalho Híbrido
1. **Frontend**: VS Code (Local) - Edição de código, IntelliSense, Copilot.
2. **Backend**: Google Colab Runtime (Remoto) - Execução Linux, Instalação do EnergyPlus, Acesso ao GCS.

## 📋 Pré-requisitos
- Extensão [Google Colab for VS Code](https://marketplace.visualstudio.com/items?itemName=google.colab) instalada.
- Conexão ativa com um Runtime do Colab (Connect to Colab).
- Acesso ao bucket GCS `eplus-colab-cloud-data` configurado.

## 🎯 Arquitetura
Este notebook trabalha diretamente com arquivos armazenados no Google Cloud Storage:
- **Modelos IDF** e **arquivos climáticos EPW** são baixados do bucket
- **Simulações** executadas localmente no runtime do Colab
- **Resultados** enviados automaticamente de volta para o bucket

Não é necessário clonar repositórios ou configurar tokens do GitHub.

---

## 🆕 Atualização: Nova Estrutura do Bucket GCS

**Data**: Fevereiro 2026

Este notebook foi atualizado para trabalhar com a nova estrutura organizada do bucket.

### 🎯 Mudanças Principais:
- ✅ **Inputs organizados**: Modelos IDF em `models/`, arquivos EPW em `weather/`
- ✅ **Outputs centralizados**: Todos os resultados em `resultados/` com timestamps
- ✅ **Stage-In atualizado**: Download automático das pastas corretas
- ✅ **Stage-Out aprimorado**: Upload apenas de outputs (ignora inputs)
- ✅ **Utilitários novos**: Funções para explorar e gerenciar arquivos no bucket
- ✅ **Simplificado**: Removida dependência de clonagem de repositórios

### 📝 Como Usar:
1. Execute as células na ordem (1 → 10)
2. Os arquivos serão baixados automaticamente de `models/` e `weather/`
3. Resultados enviados para `resultados/simulacao_vscode_{timestamp}/`
4. Use as células 9 e 10 para explorar os arquivos disponíveis no bucket

---

In [ ]:
# @title 1. Autenticação e Configuração do Projeto GCP
import os
import sys
import warnings
import re
import time
from typing import Optional

# --- Configuração ---
PROJECT_ID = 'eplus-colab-cloud'  # @param {type:"string"}

def authenticate_colab_session(project_id: str) -> None:
    """Autentica a sessão do Colab e configura o projeto GCP."""
    print(f"🔐 Configurando projeto: {project_id}")
    os.environ['GOOGLE_CLOUD_PROJECT'] = project_id

    try:
        import subprocess

        print("🚀 Iniciando autenticação...")
        print("⚠️ INSTRUÇÕES:")
        print("1. CLIQUE na URL longa que aparecerá ABAIXO (Output da Célula).")
        print("2. Faça login no navegador e copie o código.")
        print("3. Cole o código na caixa de entrada no TOPO do VS Code e pressione Enter.")

        # Executa gcloud com controle manual de I/O para garantir que a URL seja clicável
        # e não fique presa dentro do modal de input do VS Code
        # FIX: Remove credenciais antigas para evitar o prompt 'Do you want to continue (Y/n)?'
        if 'GOOGLE_APPLICATION_CREDENTIALS' in os.environ:
            print("🧹 Limpando credenciais antigas para forçar nova autenticação...")
            del os.environ['GOOGLE_APPLICATION_CREDENTIALS']

        # Configura ambiente para evitar quebra de linha na URL (truncamento)
        env = os.environ.copy()
        env['COLUMNS'] = '2000' # Largura extra para impedir quebras de linha na URL

        # Retorna ao comando padrão. O erro 'Missing scope' era causado pelo truncamento da URL.
        # Com COLUMNS=2000, a URL padrão (com escopos corretos) será gerada integralmente.
        cmd = [
            'gcloud', 'auth', 'application-default', 'login', '--no-launch-browser'
        ]

        with subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            stdin=subprocess.PIPE,
            text=True,
            bufsize=1,
            universal_newlines=True,
            env=env  # Passa o ambiente com COLUMNS=300
        ) as p:
            buffer = ""
            while True:
                char = p.stdout.read(1)
                if not char:
                    break

                sys.stdout.write(char)
                sys.stdout.flush() # Garante que o usuário veja o link
                buffer += char

                # Detecta o prompt de código
                if "verification code" in buffer.lower() or "authorization code" in buffer.lower():
                    print("\n\n⚠️  A caixa de entrada aparecerá no TOPO do VS Code agora! ⚠️")
                    print("⏳ Aguardando 2 segundos para você clicar no link...")
                    time.sleep(2)
                    user_code = input("Cole o código na caixa do topo: ")
                    p.stdin.write(user_code.strip() + "\n")
                    p.stdin.flush()
                    buffer = "" # Limpa buffer após processar o input

            p.wait()
            if p.returncode != 0:
                raise subprocess.CalledProcessError(p.returncode, cmd)

        print("✅ Usuário autenticado com sucesso (ADC).")
    except ImportError:
        print("⚠️ Runtime local detectado. Usando credenciais do sistema (ADC).")
    except Exception as e:
        print(f"❌ Erro na autenticação: {e}")
        print("   Se você cancelou o prompt, tente rodar a célula novamente.")

    # Tenta configurar via gcloud para compatibilidade com CLI
    try:
        import subprocess
        subprocess.run(['gcloud', 'config', 'set', 'project', project_id], check=True, capture_output=True)
        print(f"✅ Projeto GCP configurado: {project_id}")
    except Exception as e:
        warnings.warn(f"Não foi possível configurar gcloud CLI (não crítico): {e}")

authenticate_colab_session(PROJECT_ID)

In [ ]:
# @title 2. Diagnóstico de Recursos (CPU/RAM)
import psutil
import os

print("--- Diagnóstico do Ambiente de Execução ---")
try:
    # CPU
    cpu_count = os.cpu_count()
    print(f"🧠 CPUs Lógicas: {cpu_count}")

    # RAM
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"💾 Memória RAM Total: {ram_gb:.2f} GB")

    if ram_gb < 20:
        print("⚠️  Aviso: Runtime padrão (Standard RAM). Para grandes modelos, considere High-RAM.")
    else:
        print("✅  Runtime de Alta Memória (High-RAM) detectado.")

    # GPU (Opcional para EnergyPlus, mas bom para saber)
    gpu_info = !nvidia-smi
    gpu_info = '\n'.join(gpu_info)
    if 'failed' in gpu_info or 'not found' in gpu_info:
        print("ℹ️  Nenhuma GPU dedicada detectada (OK para EnergyPlus).")
    else:
        print("🚀 GPU Detectada (Disponível para ML/TensorFlow).")
except Exception as e:
    print(f"Erro no diagnóstico: {e}")

In [ ]:
# @title 3. Definição do Bucket GCS e Arquivos
from google.cloud import storage
from typing import List

BUCKET_NAME = 'eplus-colab-cloud-data' # @param {type:"string"}

# Estrutura do Bucket:
# gs://eplus-colab-cloud-data/
#   ├── models/          ← Arquivos IDF
#   ├── weather/         ← Arquivos EPW
#   ├── resultados/      ← Resultados das simulações
#   ├── scripts/         ← Scripts de instalação
#   └── notebooks/       ← Notebooks

# Arquivos de entrada (com paths relativos ao bucket)
IDF_FILE = 'models/5ZoneAirCooled.idf'
EPW_FILE = 'weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw'

print(f"🎯 Bucket Alvo: gs://{BUCKET_NAME}")
print(f"📁 Arquivo IDF: {IDF_FILE}")
print(f"🌤️  Arquivo EPW: {EPW_FILE}")

# Validação e inspeção do bucket usando Cloud Storage API
try:
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)

    # Verifica se o bucket existe e é acessível
    if bucket.exists():
        print("✅ Acesso ao bucket confirmado.")

        # Lista arquivos disponíveis nas pastas principais
        print("\n📋 Estrutura do bucket:")

        folders = ['models/', 'weather/']
        for folder in folders:
            print(f"\n📂 {folder}")
            blobs = list(client.list_blobs(BUCKET_NAME, prefix=folder, delimiter='/'))

            # Filtra apenas arquivos (não subpastas)
            files = [blob.name for blob in blobs if not blob.name.endswith('/') and blob.name != folder]

            if files:
                for file_path in files[:5]:  # Mostra até 5 arquivos
                    filename = file_path.split('/')[-1]
                    blob = bucket.blob(file_path)
                    # Busca metadados (tamanho)
                    blob.reload()
                    size_mb = blob.size / (1024 * 1024) if blob.size else 0
                    print(f"  • {filename} ({size_mb:.2f} MB)")

                if len(files) > 5:
                    print(f"  ... e mais {len(files) - 5} arquivo(s)")
            else:
                print("  (vazio)")

        # Verifica se os arquivos especificados existem
        print("\n🔍 Verificando arquivos de entrada:")
        idf_blob = bucket.blob(IDF_FILE)
        epw_blob = bucket.blob(EPW_FILE)

        if idf_blob.exists():
            idf_blob.reload()
            print(f"  ✅ {IDF_FILE} ({idf_blob.size / (1024 * 1024):.2f} MB)")
        else:
            print(f"  ⚠️ {IDF_FILE} não encontrado!")

        if epw_blob.exists():
            epw_blob.reload()
            print(f"  ✅ {EPW_FILE} ({epw_blob.size / (1024 * 1024):.2f} MB)")
        else:
            print(f"  ⚠️ {EPW_FILE} não encontrado!")

    else:
        print(f"❌ Bucket '{BUCKET_NAME}' não existe ou não está acessível.")

except Exception as e:
    print(f"❌ Erro ao acessar bucket: {e}")
    print("   Verifique se a autenticação (célula 1) foi executada corretamente.")

## 🛠️ Instalação do EnergyPlus v25.1.0
Instala o motor de simulação na VM Linux remota.

In [ ]:
# @title 4. Instalar EnergyPlus
from pathlib import Path
import subprocess

EPLUS_VERSION = "25.1.0"
EPLUS_URL = 'https://github.com/NREL/EnergyPlus/releases/download/v25.1.0/EnergyPlus-25.1.0-68a4a7c774-Linux-Ubuntu22.04-x86_64.run'
INSTALL_PATH = Path('/eplus')

def install_energyplus(url: str, install_path: Path) -> None:
    if install_path.exists():
        print(f"✅ EnergyPlus já instalado em {install_path}")
        return

    print(f"⬇️ Baixando EnergyPlus v{EPLUS_VERSION}...")
    installer = Path('/tmp/ep_installer.run')
    subprocess.run(['wget', '-q', '-O', str(installer), url], check=True)
    subprocess.run(['chmod', '+x', str(installer)], check=True)

    print("📦 Instalando dependências do sistema...")
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    deps = ['libxcb-icccm4', 'libxcb-image0', 'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxkbcommon-x11-0']
    subprocess.run(['apt-get', '-qq', 'install', '-y'] + deps, check=True)

    print("⚙️ Executando instalador...")
    # Cria diretório para evitar erro de desktop entry
    Path('/root/.local/share/applications').mkdir(parents=True, exist_ok=True)

    subprocess.run([str(installer), 'install', '-c', '--al', '-t', str(install_path)], check=True)
    installer.unlink() # Limpeza
    print(f"✅ Instalação concluída em {install_path}")

install_energyplus(EPLUS_URL, INSTALL_PATH)

# Adiciona ao path imediatamente para esta sessão
if str(INSTALL_PATH) not in sys.path:
    sys.path.insert(0, str(INSTALL_PATH))

In [ ]:
# @title 5. Configurar API Python (sem modulo externo)
import os
from pathlib import Path

try:
    install_path = Path(INSTALL_PATH)
    if not install_path.exists():
        raise FileNotFoundError(f"Diretorio de instalacao nao encontrado: {install_path}")

    os.environ.setdefault("EPLUS_HOME", str(install_path))
    if str(install_path) not in sys.path:
        sys.path.insert(0, str(install_path))
        print(f"✅ API do EnergyPlus vinculada ao Python: {install_path}")
    else:
        print("ℹ️  API ja configurada no path.")

    from pyenergyplus.api import EnergyPlusAPI
    api = EnergyPlusAPI()
    print(f"✅ API Carregada com Sucesso! Versao do Motor: {api.functional.ep_version()}")

except FileNotFoundError as e:
    print(f"❌ {e}")
    print("   Verifique se a celula 4 (Instalar EnergyPlus) rodou corretamente.")
except Exception as e:
    print(f"❌ Erro fatal na API: {e}")

## ▶️ Execução da Simulação
1. **Stage-In**: Download do GCS para `/tmp`.
2. **Run**: Execução via `pyenergyplus`.
3. **Stage-Out**: Upload dos resultados para o GCS.

In [ ]:
# @title 6. Stage-In (Download Inputs)
from google.cloud import storage

WORK_DIR = Path('/tmp/energyplus_sim')
WORK_DIR.mkdir(exist_ok=True)

def download_inputs(bucket_name: str, files: dict[str, str], dest_dir: Path) -> dict[str, str]:
    """
    Download arquivos do GCS para o diretório local.

    Args:
        bucket_name: Nome do bucket GCS
        files: Dicionário {tipo: path_no_bucket} ex: {'idf': 'models/file.idf'}
        dest_dir: Diretório de destino local

    Returns:
        Dicionário com paths locais dos arquivos baixados
    """
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    local_files = {}
    print(f"⬇️ Baixando arquivos de gs://{bucket_name}...")

    for file_type, gcs_path in files.items():
        blob = bucket.blob(gcs_path)
        # Salva apenas com o nome do arquivo (sem a estrutura de pastas)
        filename = Path(gcs_path).name
        dest = dest_dir / filename

        blob.download_to_filename(str(dest))
        local_files[file_type] = str(dest)
        print(f"  ✓ {gcs_path} → {filename}")

    return local_files

# Download dos arquivos de entrada
input_files = {
    'idf': IDF_FILE,
    'epw': EPW_FILE
}

local_paths = download_inputs(BUCKET_NAME, input_files, WORK_DIR)
local_idf = local_paths['idf']
local_epw = local_paths['epw']

print(f"\n📂 Arquivos locais prontos:")
print(f"  IDF: {local_idf}")
print(f"  EPW: {local_epw}")

In [ ]:
# @title 7. Rodar Simulação (API)
api = EnergyPlusAPI()
state = api.state_manager.new_state()

args = [
    '-d', str(WORK_DIR),
    '-w', local_epw,
    local_idf
]

print(f"🚀 Iniciando simulação de {IDF_FILE}...")
exit_code = api.runtime.run_energyplus(state, args)

if exit_code == 0:
    print("\n✅ SIMULAÇÃO BEM-SUCEDIDA!")
else:
    print(f"\n❌ Falha na simulação. Código: {exit_code}")

api.state_manager.delete_state(state)

In [ ]:
# @title 8. Stage-Out (Upload Resultados)
from datetime import datetime

def upload_results(bucket_name: str, source_dir: Path, prefix: str) -> int:
    """
    Upload resultados para o bucket GCS.

    Args:
        bucket_name: Nome do bucket GCS
        source_dir: Diretório local com os resultados
        prefix: Prefixo no bucket (ex: 'resultados/simulacao_20260206_120000')

    Returns:
        Número de arquivos enviados
    """
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    print(f"⬆️ Enviando resultados para gs://{bucket_name}/{prefix}...")
    count = 0

    for file_path in source_dir.iterdir():
        if file_path.is_file() and not file_path.name.endswith(('.idf', '.epw')):
            # Ignora inputs, envia apenas outputs do EnergyPlus
            blob = bucket.blob(f"{prefix}/{file_path.name}")
            blob.upload_from_filename(str(file_path))
            print(f"  ✓ {file_path.name}")
            count += 1

    return count

if exit_code == 0:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Nova estrutura: resultados/simulacao_vscode_{timestamp}/
    gcs_prefix = f"resultados/simulacao_vscode_{timestamp}"

    files_uploaded = upload_results(BUCKET_NAME, WORK_DIR, gcs_prefix)

    print(f"\n✅ {files_uploaded} arquivos de resultado enviados com sucesso!")
    print(f"\n🔗 Resultados disponíveis em:")
    print(f"   GCS Browser: https://console.cloud.google.com/storage/browser/{BUCKET_NAME}/{gcs_prefix}")
    print(f"   gsutil: gsutil ls -lh gs://{BUCKET_NAME}/{gcs_prefix}/")
else:
    print("⚠️ Simulação falhou, upload cancelado.")

In [ ]:
# @title 9. Visualizar Relatório HTML
from IPython.display import HTML, display

report = WORK_DIR / 'eplustbl.htm'
if report.exists():
    # Lê apenas o início do arquivo para evitar travar o navegador com tabelas gigantes
    with open(report, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read(5000) # Primeiros 5000 caracteres (Cabeçalho + Resumo)

    print("📄 Visualizando prévia do relatório (truncado):")
    display(HTML(content + "...<br><br><b>⚠️ Relatório completo disponível no Bucket GCS (link acima).</b>"))
else:
    print("⚠️ Relatório HTML não encontrado.")

In [ ]:
# @title 10. Visualização Rápida (Temperaturas)
import pandas as pd
import matplotlib.pyplot as plt

csv_path = WORK_DIR / 'eplusout.csv'

# Gera o Gráfico
if csv_path.exists():
    print("📊 Gerando gráfico de temperaturas...")
    try:
        # Lê o CSV ignorando erros de formatação comuns do E+
        df = pd.read_csv(csv_path)

        # Busca colunas que contenham 'Temperature' e 'Zone' (Case insensitive)
        temp_cols = [c for c in df.columns if 'Temperature' in c and 'Zone' in c]

        if temp_cols:
            plt.figure(figsize=(15, 6))
            # Plota as primeiras 5 zonas encontradas
            for col in temp_cols[:5]:
                plt.plot(df[col], label=col)

            plt.title("Perfil de Temperaturas das Zonas")
            plt.xlabel("Time Step")
            plt.ylabel("Temperatura (°C)")
            plt.legend(loc='best', fontsize='small')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print("ℹ️ Nenhuma coluna de temperatura de zona encontrada.")
    except Exception as e:
        print(f"❌ Erro ao plotar: {e}")
else:
    print("⚠️ Arquivo de resultados não encontrado.")

In [ ]:
# @title 11. Utilitários para Explorar o Bucket
from google.cloud import storage
from typing import List

def list_bucket_structure(bucket_name: str) -> None:
    """Lista a estrutura de pastas do bucket."""
    print(f"📁 Estrutura do bucket gs://{bucket_name}:\n")

    folders = ['models/', 'weather/', 'resultados/', 'scripts/', 'notebooks/']

    for folder in folders:
        print(f"\n📂 {folder}")
        result = subprocess.run(
            ['gsutil', 'ls', f'gs://{bucket_name}/{folder}'],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            files = [line.strip() for line in result.stdout.split('\n') if line.strip()]
            if files:
                for file in files[:10]:  # Mostra até 10 arquivos
                    filename = file.split('/')[-1]
                    if filename:
                        print(f"  • {filename}")
                if len(files) > 10:
                    print(f"  ... e mais {len(files) - 10} arquivos")
            else:
                print("  (vazio)")
        else:
            print(f"  ⚠️ Não foi possível listar")

def list_available_models(bucket_name: str) -> List[str]:
    """Lista modelos IDF disponíveis."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix='models/')
    models = [blob.name for blob in blobs if blob.name.endswith('.idf')]

    print("📋 Modelos IDF disponíveis:")
    for model in models:
        print(f"  • {model}")

    return models

def list_available_weather(bucket_name: str) -> List[str]:
    """Lista arquivos EPW disponíveis."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix='weather/')
    weather_files = [blob.name for blob in blobs if blob.name.endswith('.epw')]

    print("🌤️  Arquivos de clima disponíveis:")
    for wf in weather_files:
        print(f"  • {wf}")

    return weather_files

def list_recent_results(bucket_name: str, limit: int = 5) -> None:
    """Lista os últimos resultados de simulação."""
    result = subprocess.run(
        ['gsutil', 'ls', f'gs://{bucket_name}/resultados/'],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        folders = [line.strip() for line in result.stdout.split('\n') if 'simulacao_' in line]
        folders.sort(reverse=True)  # Mais recentes primeiro

        print(f"📊 Últimas {min(limit, len(folders))} simulações:")
        for folder in folders[:limit]:
            sim_name = folder.rstrip('/').split('/')[-1]
            print(f"  • {sim_name}")
            print(f"    {folder}")
    else:
        print("⚠️ Não foi possível listar resultados")

# Descomente as funções que deseja executar:
print("💡 Funções disponíveis:")
print("  • list_bucket_structure(BUCKET_NAME)")
print("  • list_available_models(BUCKET_NAME)")
print("  • list_available_weather(BUCKET_NAME)")
print("  • list_recent_results(BUCKET_NAME)")
print("\nExemplo de uso:")
print("  list_bucket_structure(BUCKET_NAME)")


## 📚 Estrutura do Bucket GCS

O bucket `eplus-colab-cloud-data` está organizado da seguinte forma:

```
gs://eplus-colab-cloud-data/
├── models/              # Arquivos IDF (modelos de edificações)
│   └── 5ZoneAirCooled.idf
├── weather/             # Arquivos EPW (dados climáticos)
│   └── USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw
├── resultados/          # Outputs das simulações (organizados por timestamp)
│   ├── simulacao_vscode_20260206_120000/
│   └── simulacao_cloudshell_20260206_195217/
├── scripts/             # Scripts de instalação e automação
│   └── install_energyplus.sh
└── notebooks/           # Notebooks arquivados
    └── _legacy/
```

### 🔄 Fluxo de Trabalho

1. **Modelos e Clima**: Armazenados em `models/` e `weather/`
2. **Execução**: Notebook baixa arquivos, executa simulação localmente
3. **Resultados**: Enviados automaticamente para `resultados/simulacao_vscode_{timestamp}/`

### 📝 Como Adicionar Novos Arquivos

```bash
# Upload de novo modelo IDF
gsutil cp meu_modelo.idf gs://eplus-colab-cloud-data/models/

# Upload de arquivo climático
gsutil cp cidade.epw gs://eplus-colab-cloud-data/weather/

# Listar resultados
gsutil ls -lh gs://eplus-colab-cloud-data/resultados/
```

In [ ]:
# @title 12. [EXEMPLO] Explorar Arquivos Disponíveis
# Execute esta célula para ver todos os arquivos disponíveis no bucket

print("="*60)
print("🔍 EXPLORANDO O BUCKET GCS")
print("="*60)

# Lista modelos disponíveis
print("\n")
models = list_available_models(BUCKET_NAME)

# Lista arquivos de clima disponíveis
print("\n")
weather = list_available_weather(BUCKET_NAME)

# Lista resultados recentes
print("\n")
list_recent_results(BUCKET_NAME, limit=10)

print("\n" + "="*60)
print(f"✅ Total: {len(models)} modelos, {len(weather)} arquivos climáticos")
print("="*60)

## 📚 Guia Completo: Adicionar Novos Arquivos ao Bucket

### 🎯 Objetivo
Adicionar novos modelos IDF e arquivos climáticos (EPW) ao bucket GCS para utilização em futuras simulações.

### 📋 Pré-requisitos
- **gsutil** instalado e configurado (já feito neste notebook)
- **gcloud CLI** autenticado (feito na Cell 1)
- Acesso ao bucket `gs://eplus-colab-cloud-data`

---

## 🔧 Método 1: Upload via gsutil (Recomendado - Local)

### Para Modelos IDF:
```bash
# Enviar um único modelo
gsutil cp meu_modelo.idf gs://eplus-colab-cloud-data/models/

# Enviar vários modelos de uma pasta
gsutil -m cp /caminho/local/modelos/*.idf gs://eplus-colab-cloud-data/models/

# Copiar com substituição (caso já exista)
gsutil -m cp -r /caminho/local/modelos/* gs://eplus-colab-cloud-data/models/
```

### Para Arquivos Climáticos (EPW):
```bash
# Enviar um único arquivo climático
gsutil cp chicago.epw gs://eplus-colab-cloud-data/weather/

# Enviar vários arquivos climáticos
gsutil -m cp /caminho/local/climas/*.epw gs://eplus-colab-cloud-data/weather/

# Copiar recursivamente (mantém estrutura de pastas)
gsutil -m cp -r /caminho/local/weather/* gs://eplus-colab-cloud-data/weather/
```

### Para Resultados/Scripts:
```bash
# Upload de scripts
gsutil cp meu_script.py gs://eplus-colab-cloud-data/scripts/

# Upload de notebooks
gsutil cp meu_notebook.ipynb gs://eplus-colab-cloud-data/notebooks/
```

**Flags úteis:**
- `-m`: Upload paralelo (mais rápido para múltiplos arquivos)
- `-r`: Recursivo (para pastas)
- `-C`: Ignora erros e continua
- `-h`: Headers HTTP personalizados (ex: metadados)

---

## 🐍 Método 2: Upload via Python (No Notebook)

Você pode executar o código abaixo diretamente no notebook para fazer upload:

```python
from google.cloud import storage
from pathlib import Path

PROJECT_ID = 'eplus-colab-cloud'
BUCKET_NAME = 'eplus-colab-cloud-data'

def upload_file_to_gcs(local_path: str, gcs_folder: str):
    """Faz upload de um arquivo para o bucket GCS."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    
    local_file = Path(local_path)
    if not local_file.exists():
        print(f"❌ Arquivo não encontrado: {local_path}")
        return
    
    # Cria o path no bucket
    gcs_path = f"{gcs_folder.rstrip('/')}/{local_file.name}"
    blob = bucket.blob(gcs_path)
    
    # Upload
    print(f"⬆️ Enviando {local_file.name}...")
    blob.upload_from_filename(str(local_file))
    print(f"✅ Enviado para gs://{BUCKET_NAME}/{gcs_path}")

# Exemplos de uso:
# upload_file_to_gcs("meu_modelo.idf", "models")
# upload_file_to_gcs("chicago.epw", "weather")
```

---

## 📂 Estrutura Esperada do Bucket

Mantenha a estrutura organizada:

```
gs://eplus-colab-cloud-data/
├── models/
│   ├── 5ZoneAirCooled.idf          ← Modelos IDF
│   ├── Office_Medium.idf
│   └── Hospital_Large.idf
├── weather/
│   ├── USA_IL_Chicago-OHare...epw  ← Arquivos climáticos
│   ├── USA_CA_Los_Angeles...epw
│   └── BRA_RJ_Rio_de_Janeiro...epw
├── resultados/
│   ├── simulacao_vscode_20260206_211001/
│   └── simulacao_vscode_20260206_180000/
├── scripts/
│   └── install_energyplus.sh
└── notebooks/
    └── EnergyPlus_VS_Code_Colab.ipynb
```

---

## 🔍 Verificar Uploads

### Via gsutil:
```bash
# Listar arquivos em models/
gsutil ls -lh gs://eplus-colab-cloud-data/models/

# Listar arquivos em weather/
gsutil ls -lh gs://eplus-colab-cloud-data/weather/

# Verificar tamanho total do bucket
gsutil du -sh gs://eplus-colab-cloud-data/
```

### Via Python (no notebook):
```python
def list_files_in_folder(bucket_name: str, folder: str):
    client = storage.Client(project=PROJECT_ID)
    blobs = client.list_blobs(bucket_name, prefix=folder)
    
    print(f"📂 Arquivos em {folder}:")
    for blob in blobs:
        size_mb = blob.size / (1024 * 1024)
        print(f"  • {blob.name} ({size_mb:.2f} MB)")

# Exemplos:
# list_files_in_folder(BUCKET_NAME, "models")
# list_files_in_folder(BUCKET_NAME, "weather")
```

---

## 🔄 Workflow para Novas Simulações

### Passo 1: Preparar Files Localmente
1. Obter modelo IDF (de `energyplus-weather.run` ou criar novo)
2. Obter arquivo climático EPW (de `energyplus-weather.run`)
3. Validar estrutura dos arquivos

### Passo 2: Upload para Bucket
```bash
gsutil cp meu_novo_modelo.idf gs://eplus-colab-cloud-data/models/
gsutil cp cidade_nova.epw gs://eplus-colab-cloud-data/weather/
```

### Passo 3: Atualizar Cell 3 do Notebook
```python
IDF_FILE = 'models/meu_novo_modelo.idf'
EPW_FILE = 'weather/cidade_nova.epw'
```

### Passo 4: Executar Pipeline (Cells 6-8)
- Cell 6: Download dos novos arquivos
- Cell 7: Executar simulação
- Cell 8: Fazer upload dos resultados

---

## 💡 Dicas & Boas Práticas

### ✅ O Que Fazer:
- **Organizar por pasta**: Manter `models/`, `weather/`, `resultados/` separados
- **Nomes descritivos**: Use nomes claros (ex: `Chicago_TMY3.epw` em vez de `weather1.epw`)
- **Versionamento**: Se atualizar arquivo, adicione timestamp (ex: `modelo_v2_20260206.idf`)
- **Documentação**: Crie um README.md no bucket com metadados dos arquivos
- **Compactação**: Para uploads grandes, comprimir com `.tar.gz` antes

### ❌ O Que Evitar:
- Não deletar arquivos de entrada enquanto há simulações rodando
- Não usar nomes com caracteres especiais ou espaços
- Não fazer upload de arquivos intermediários (binários, caches)
- Não misturar inputs com outputs na mesma pasta

---

## 🎓 Exemplo Completo: Adicionar Novo Modelo

**Cenário**: Você tem um novo modelo chamado `Hotel_5Star.idf` e dados climáticos de `Miami.epw`

### Via Terminal Local:
```bash
# Faça login no GCP (se não estiver)
gcloud auth login

# Configure projeto
gcloud config set project eplus-colab-cloud

# Upload dos arquivos
gsutil cp Hotel_5Star.idf gs://eplus-colab-cloud-data/models/
gsutil cp Miami_TMY3.epw gs://eplus-colab-cloud-data/weather/

# Verificar upload
gsutil ls -lh gs://eplus-colab-cloud-data/models/Hotel_5Star.idf
gsutil ls -lh gs://eplus-colab-cloud-data/weather/Miami_TMY3.epw
```

### Via Notebook (Cell 6-8):
```python
# Editar Cell 3
IDF_FILE = 'models/Hotel_5Star.idf'
EPW_FILE = 'weather/Miami_TMY3.epw'

# Executar Cells 6, 7, 8 normalmente
# Os resultados irão para:
# gs://eplus-colab-cloud-data/resultados/simulacao_vscode_TIMESTAMP/
```

---

## 🔗 Recursos Úteis

- **Modelos IDF**: [EnergyPlus Example Files](https://energyplus.net/weather/download)
- **Dados Climáticos**: [IWEC Weather Data](https://energyplus.net/weather)
- **Documentação EnergyPlus**: [energyplus.net](https://energyplus.net/)
- **Google Cloud Storage**: [cloud.google.com/storage/docs](https://cloud.google.com/storage/docs)

---

## 📞 Troubleshooting

| Problema | Solução |
|----------|---------|
| `gsutil: command not found` | Instale Google Cloud SDK: `curl https://sdk.cloud.google.com \| bash` |
| `AccessDenied` ao fazer upload | Verifique permissões: `gsutil acl ch -u $(gcloud config get-value account):O gs://eplus-colab-cloud-data` |
| Arquivo não encontrado após upload | Aguarde 1-2 minutos (cache de listagem) ou use `gsutil ls` com `--stat` |
| Upload muito lento | Use flag `-m` para paralelizar ou comprima arquivo primeiro |

